We train a DenseNet121 on the NIH ChestXray dataset using the Infiltration finding as a proxy for tuberculosis. The goal is a model we can then test on other populations, so we train on NIH only and keep a held-out split for a fair home score. This notebook runs on the Kaggle T4 GPU.

In [ ]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from collections import Counter

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:",device)
print("torch:", torch.__version__, "| pandas:", pd.__version__)

In [ ]:
df=pd.read_csv("/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv")
print(df.head(100))
print("=============")
print(df.sum())
print("=============")
print(df["Finding Labels"].value_counts().head(10))
print("=============")
pos=df[df["Finding Labels"].str.contains("Infiltration")].copy()
neg=df[df["Finding Labels"] == "No Finding"].copy()
print("pos:", len(pos), "| neg:", len(neg), "| ratio 1:", round(len(pos) / len(neg), 2))
print("=============")

The data. We read the NIH metadata file and keep two classes only, images that show Infiltration and images with no finding. Everything else is dropped, which leaves about 80,000 images with a roughly one to three ratio of positives to negatives.

In [ ]:
patients=df["Patient ID"].unique()
g = torch.Generator().manual_seed(42)
patients = patients[torch.randperm(len(patients), generator=g)]
n=len(patients)
train_pat=set(patients[:int(n*0.70)])
dev_pat=set(patients[int(n*0.70):int(n*0.85)])
test_pat=set(patients[int(n*0.85):])
def tag(p):
    if p in train_pat:return "train"
    if p in dev_pat:return "dev"
    else :return "test"
df["split"] = df["Patient ID"].map(tag)
train_df=df[df["split"]=="train"]
dev_df=df[df["split"]=="dev"]
test_df=df[df["split"]=="test"]
print("=======train dataframe========")
print(train_df.head())
print("========dev dataframe========")
print(dev_df.head())
print("=======test dataframe=========")
print(test_df.head())
print("train∩dev:", len(set(train_df["Patient ID"]) & set(dev_df["Patient ID"])))
print("train∩test:", len(set(train_df["Patient ID"]) & set(test_df["Patient ID"])))
print("dev∩test:", len(set(dev_df["Patient ID"]) & set(test_df["Patient ID"])))

Patient split. We split by patient ID so the same person never lands in more than one set. That keeps the dev and test scores honest, because images from one patient are not independent.

In [ ]:
class ChestXRayDataset(Dataset):
    def __init__(self,df,img_dir,transform):
        self.df=df
        self.img_dir=img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df) 
    def __getitem__(self,idx):
        row=self.df.iloc[idx]
        path = f"{self.img_dir}/{row['Image Index']}"
        img = Image.open(path).convert("L")
        img = self.transform(img)
        label = 1 if "Infiltration" in row["Finding Labels"] else 0
        return img, torch.tensor(label, dtype=torch.float32)

In [ ]:
IMG_DIR="/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomRotation(degrees=15),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
simple_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
train_ds=ChestXRayDataset(train_df,IMG_DIR,train_transform)
dev_ds=ChestXRayDataset(dev_df,IMG_DIR,simple_transform)
test_ds=ChestXRayDataset(test_df,IMG_DIR,simple_transform)

In [ ]:
train_dloader = DataLoader(train_ds, batch_size=32, shuffle=True,num_workers=2)
dev_dloader   = DataLoader(dev_ds,   batch_size=32, shuffle=False,num_workers=2)
test_dloader  = DataLoader(test_ds,  batch_size=32, shuffle=False,num_workers=2)

In [ ]:
model=models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
model.classifier=nn.Linear(1024,1)
model=model.to(device)

Model setup. We load a pretrained DenseNet121 and swap the last layer for a single output, one score for our two classes. All weights are trainable.

In [ ]:
num_epochs = 25
patience   = 6
best_dev_auc = 0.0
no_improve = 0
def manual_auc(labels, probs):
    y = torch.tensor(labels, dtype=torch.float32)
    p = torch.tensor(probs, dtype=torch.float32)
    p_pos = p[y == 1]
    p_neg = p[y == 0]
    higher = (p_pos.unsqueeze(1) > p_neg.unsqueeze(0)).sum().float()
    tied   = (p_pos.unsqueeze(1) == p_neg.unsqueeze(0)).sum().float()
    return ((higher + 0.5 * tied) / (len(p_pos) * len(p_neg))).item()
n_pos = int((train_df["Finding Labels"].str.contains("Infiltration")).sum())
n_neg = len(train_df) - n_pos
pos_weight = torch.tensor([n_neg / n_pos]).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
with open("/kaggle/working/training_log.csv", "w") as f:
    f.write("epoch,train_loss,dev_auc,lr\n")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_dloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze(1)
        loss = loss_fn(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
    model.eval()
    dev_probs, dev_labels = [], []
    with torch.no_grad():
        for img, lbl in dev_dloader:
            img = img.to(device)
            p = torch.sigmoid(model(img)).cpu()
            dev_probs.extend(p.numpy().ravel())
            dev_labels.extend(lbl.numpy())
    dev_auc = manual_auc(dev_labels, dev_probs)
    print(f"Epoch {epoch+1} | train loss: {running_loss/len(train_dloader):.4f} | dev AUC: {dev_auc:.4f}")
    with open("/kaggle/working/training_log.csv", "a") as f:
        f.write(f"{epoch+1},{running_loss/len(train_dloader):.4f},{dev_auc:.4f},{scheduler.get_last_lr()[0]:.6f}\n")
    if dev_auc > best_dev_auc:
        best_dev_auc = dev_auc
        torch.save(model.state_dict(), "/kaggle/working/best_model.pt")
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping.")
            break
    scheduler.step()

This is the training loop. We use a weighted loss because the classes are imbalanced, add weight decay and a cosine decay schedule on the learning rate, log one row per epoch, and save the best model by development AUC with early stopping.

In [ ]:
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
model.classifier = nn.Linear(1024, 1)
model.load_state_dict(torch.load("/kaggle/working/best_model.pt"))
model=model.to(device)
model.eval()

Evaluation. We reload the saved best model and run it over the held-out test split to get the home AUC, then save the predictions and plot the ROC curve.

In [ ]:
test_probs, test_labels = [], []                                         
with torch.no_grad():                                                    
    for img, lbl in test_dloader:                                        
        img = img.to(device)                                             
        p = torch.sigmoid(model(img)).cpu()                              
        test_probs.extend(p.numpy().ravel())                             
        test_labels.extend(lbl.numpy())                                  
probs = torch.tensor(test_probs)                                         
labels = torch.tensor(test_labels)                                       

In [ ]:
print("n_test:", len(labels), "| positives:", int(labels.sum()), "| negatives:", int((labels == 0).sum()))
print("probs range:", round(probs.min().item(), 4), "-", round(probs.max().item(), 4))
auc = manual_auc(test_labels, test_probs)
preds = (probs >= 0.5).float()
tp = ((preds == 1) & (labels == 1)).sum().item()
fp = ((preds == 1) & (labels == 0)).sum().item()
fn = ((preds == 0) & (labels == 1)).sum().item()
tn = ((preds == 0) & (labels == 0)).sum().item()
prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
print(f"AUC: {auc:.4f} | TP:{tp} FP:{fp} FN:{fn} TN:{tn}")
print(f"Precision: {prec:.3f} | Recall: {rec:.3f} | F1: {f1:.3f}")
print("preds==1 count:", int(preds.sum()))

In [ ]:
print(f"Test AUC: {auc:.4f}")
print(f"TP:{tp} FP:{fp} FN:{fn} TN:{tn}")
print(f"Infiltration → precision {prec:.3f} | recall {rec:.3f} | F1 {f1:.3f}")

In [ ]:
test_df[["Image Index"]].assign(true_label=test_labels, prob=test_probs).to_csv(
    "/kaggle/working/test_predictions.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
thresholds = torch.linspace(0, 1, 200)
tprs, fprs = [], []
for t in thresholds:
    preds = (probs >= t).float()
    tp = ((preds == 1) & (labels == 1)).sum()
    fn = ((preds == 0) & (labels == 1)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    tn = ((preds == 0) & (labels == 0)).sum()
    tprs.append(tp / (tp + fn))
    fprs.append(fp / (fp + tn))
plt.figure(figsize=(5, 5))
plt.plot(fprs, tprs, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Random (0.5)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC — NIH held-out test")
plt.legend()
plt.show()

In [ ]:
import os, time
print(os.listdir("/kaggle/working"))
if "best_model.pt" in os.listdir("/kaggle/working"):
    print("modified:", time.ctime(os.path.getmtime("/kaggle/working/best_model.pt")))

In [ ]:
import zipfile, os
plt.savefig("/kaggle/working/roc_nih.png", dpi=150, bbox_inches="tight")
with zipfile.ZipFile("/kaggle/working/pp1_artifacts.zip", "w") as z:
    for f in ["best_model.pt", "training_log.csv", "test_predictions.csv", "roc_nih.png"]:
        p = f"/kaggle/working/{f}"
        if os.path.exists(p):
            z.write(p, arcname=f)
            print("packed:", f)
        else:
            print("MISSING:", f)
print(os.listdir("/kaggle/working"))

We save the ROC figure and pack the model, the training log, the test predictions, and the figure into a zip file so we can reuse them later.